In [1]:
%%time
import pandas as pd
import numpy as np
import os
from scipy.stats import ttest_1samp
import warnings
# Ignores math warnings when features are all exactly 0.0 (zero variance)
warnings.filterwarnings("ignore", category=RuntimeWarning)

datasets = ["Dataset_I", "Dataset_II", "Dataset_III", "Dataset_IV"]
versions = ["v0", "v1", "v2", "v3", "v4", "v5", "v6", "v7", "v8", "v9"]

for name in datasets:
    all_coefs = []
    
    # 1. Load the 10 vertical CSVs for this specific dataset
    for v in versions:
        file_path = f"HiLASSO_Coefs/{name}_{v}_HiLASSO_coefs.csv"
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            all_coefs.append(df['Coefficient'].values)
            
    if not all_coefs:
        print(f"Could not find CSVs for {name}. Skipping...")
        continue

    # 2. Stack them into a matrix (10 rows of seeds, P columns of features)
    coef_matrix = np.vstack(all_coefs)
    
    # 3. THE T-TEST MATH
    # We test if the mean across the 10 runs is statistically different from 0.0
    mean_coefs = np.mean(coef_matrix, axis=0)
    t_stats, p_values = ttest_1samp(coef_matrix, popmean=0.0, axis=0)
    
    # 4. Format the final output
    features = df['Feature'].values 
    results_df = pd.DataFrame({
        'Feature': features,
        'Mean_Coefficient': mean_coefs,
        'P_Value': p_values
    })
    
    # If a feature was heavily penalized and always exactly 0.0, the t-test returns NaN. 
    # We fill those with 1.0 (meaning 100% chance it is just noise).
    results_df['P_Value'] = results_df['P_Value'].fillna(1.0)
    
    # 5. Sort to find the "Top Genes" your teammate asked for
    # We sort by the absolute magnitude of the mean coefficient
    results_df['Abs_Mean'] = np.abs(results_df['Mean_Coefficient'])
    results_df = results_df.sort_values(by='Abs_Mean', ascending=False).drop(columns=['Abs_Mean'])
    
    # Save the final ranked answer key!
    results_df.to_csv(f"results/{name}_Final_T_Test_Ranks.csv", index=False)
    print(f"Generated T-test rankings for {name}! Top feature: {results_df.iloc[0]['Feature']}")

print("\nAll datasets processed. Your mentor's T-test files are ready!")

Generated T-test rankings for Dataset_I! Top feature: Beta_0
Generated T-test rankings for Dataset_II! Top feature: Beta_27
Generated T-test rankings for Dataset_III! Top feature: Beta_27
Generated T-test rankings for Dataset_IV! Top feature: Beta_25

All datasets processed. Your mentor's T-test files are ready!
CPU times: user 12.1 s, sys: 116 ms, total: 12.2 s
Wall time: 1.02 s
